# Laboratorium 2: reguła Bayesa i metoda k najbliższych sąsiadów

Karta laboratorium: <https://rodakt.github.io/MAD/laby/02-knn.html>

Treść zadań jest w książce, tutaj jest tylko szkielet kodu. Miejsca do uzupełnienia oznaczono `...`.
Notebook otwarty z linku na karcie laboratorium nie jest zapisany na Twoim Dysku: po ponownym kliknięciu linku zobaczysz wersję wyjściową.
Na początku zajęć zapisz kopię (*Plik → Zapisz kopię na Dysku*) i pracuj na niej.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from scipy.stats import norm

rng = np.random.default_rng(2026)

## Część 1. $k$-NN ręcznie (Zadanie 3.1)

Najpierw liczymy $k$-NN sami, w numpy, i sprawdzamy wyniki z tablicy. Potem to samo liczymy biblioteką scikit-learn.

Wiersz macierzy `X_tr` to jedna obserwacja ze zbioru uczącego (ang. *training set*, stąd `tr`), kolumny to cechy $x_1$, $x_2$.
Wektor `y_tr` zawiera klasy obserwacji, a `y_reg_tr` -- wyniki liczbowe $y$ do regresji.
Punkt $\mathbf{x}=(1,3)$, dla którego robimy predykcję, to wektor `x`.

In [ ]:
X_tr = np.array([[1, 1], [2, 1], [4, 3], [5, 4], [2, 3], [3, 2]], dtype=float)
y_tr = np.array(["A", "A", "B", "B", "B", "A"])   # klasa
y_reg_tr = np.array([2.0, 2.5, 5.0, 6.5, 4.0, 3.5])   # wynik liczbowy do regresji

x = np.array([1, 3], dtype=float)

### 1.1. Odległości i sąsiedzi

`X_tr - x` odejmuje wektor `x` od każdego wiersza macierzy `X_tr` (numpy sam powtarza `x` dla każdego wiersza;
nazywa się to *broadcasting*). Wiersz $i$ wyniku to $\mathbf{x}_i-\mathbf{x}$. Kwadraty różnic sumujemy w każdym wierszu
(`axis=1`) i pierwiastkujemy: dostajemy wektor sześciu odległości euklidesowych.

`np.argsort(d)` zwraca indeksy, które ustawiają `d` rosnąco; pierwsze $k$ z nich to $N_k(\mathbf{x})$.
Przy `kind="stable"` z równych odległości pierwsza jest obserwacja o niższym numerze.
Uwaga: indeksy w Pythonie zaczynają się od 0, więc obserwacja $i$ z tabeli w książce ma indeks $i-1$.

In [ ]:
diffs = X_tr - x   # kształt (6, 2): wiersz i to x_i - x
d = ...              # odległości euklidesowe, kształt (6,)
print(np.round(d, 3))

order = np.argsort(d, kind="stable")
print("N_1:", order[:1], "  N_3:", order[:3])

Czy zbiór $N_3(\mathbf{x})$ jest wyznaczony jednoznacznie? Sprawdź w wypisanych odległościach, czy trzeci i czwarty sąsiad
nie leżą w tej samej odległości.

### 1.2. Predykcje

`y_reg_tr[idx]` dla wektora indeksów `idx` to wyniki sąsiadów (indeksowanie tablicą, jak w laboratorium 1); ich średnia
to predykcja regresji. `y_tr[idx] == "B"` to wektor wartości logicznych, a jego średnia -- frakcja sąsiadów
z klasą B, czyli $\widehat\eta_{\mathrm B}(\mathbf{x})$.

In [ ]:
for k in (1, 3):
    idx = order[:k]   # indeksy k najbliższych sąsiadów
    eta_B = ...           # frakcja sąsiadów z klasą B
    y_hat = ...           # średnia wyników sąsiadów
    print(f"k = {k}:  eta_B = {eta_B:.3f},  klasa = {'B' if eta_B > 0.5 else 'A'},  regresja = {y_hat:.3f}")

Punkt d): powtórz obliczenia z odległością miejską, czyli sumą wartości bezwzględnych różnic (`np.abs`); wystarczy
skopiować kod z 1.1 i 1.2 i zmienić wzór na `d`. Ile obserwacji leży w odległości równej odległości trzeciego sąsiada?
Zmień kolejność wierszy w zbiorze uczącym (np. obserwację 3 na początek: `perm = [2, 0, 1, 3, 4, 5]`,
potem `X_tr[perm]`, `y_tr[perm]`) i sprawdź, czy zmienia się predykcja klasy dla $k=3$.

In [ ]:
...

### 1.3. To samo w scikit-learn

Model tworzymy z hiperparametrem $k$ (`n_neighbors`), „uczymy” metodą `fit` (dla $k$-NN to tylko zapamiętanie danych)
i pytamy o predykcję metodą `predict`. scikit-learn oczekuje nowych punktów w postaci macierzy, z jednym wierszem
na punkt, więc wektor `x` zamieniamy na macierz $1\times 2$: `x.reshape(1, -1)`.
`predict_proba` zwraca $\widehat\eta$ dla klas w kolejności `klf.classes_`,
a `kneighbors` -- odległości do sąsiadów i ich indeksy. Porównaj z wynikami z 1.1 i 1.2.

In [ ]:
klf = KNeighborsClassifier(n_neighbors=3).fit(X_tr, y_tr)
reg = KNeighborsRegressor(n_neighbors=3).fit(X_tr, y_reg_tr)

X_new = x.reshape(1, -1)   # macierz 1 x 2: jeden nowy punkt
print(klf.kneighbors(X_new))
print(klf.classes_, klf.predict_proba(X_new), klf.predict(X_new))
print(reg.predict(X_new))

## Część 2. Rola $k$

Wracamy do przykładu z Zadania 1.14: $Y=\sin(2\pi X)+\varepsilon$, $X$ jednostajny na $[0,1]$,
$\varepsilon\sim N(0,\,0{,}3^2)$, $n=30$. Zamiast wielomianów dopasowujemy regresję $k$-NN
dla różnych $k$. Ryzyko, jak w laboratorium 1, przybliżamy średnią stratą kwadratową na dużej nowej próbie.

scikit-learn oczekuje cech w postaci macierzy $n\times p$, a tu jest jedna cecha, stąd `X.reshape(-1, 1)`.

In [ ]:
sigma = 0.3

def generate_data(n):
    X = rng.uniform(0, 1, size=n)
    Y = np.sin(2 * np.pi * X) + rng.normal(0, sigma, size=n)
    return X.reshape(-1, 1), Y

X_tr, Y_tr = generate_data(30)
X_te, Y_te = generate_data(100_000)

In [ ]:
grid = np.linspace(0, 1, 1000).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, k in zip(axes, [1, 5, 30]):
    reg = KNeighborsRegressor(n_neighbors=k).fit(X_tr, Y_tr)
    ax.scatter(X_tr, Y_tr, s=12, color="0.5")
    ax.plot(grid, np.sin(2 * np.pi * grid), "k--", lw=1)
    ax.plot(grid, reg.predict(grid), lw=1.6)
    ax.set_title(f"$k = {k}$")
plt.show()

Dla każdego $k$ oblicz błąd na zbiorze uczącym (`X_tr`, `Y_tr`) i oszacuj ryzyko średnią stratą na dużej nowej
próbie (`X_te`, `Y_te`), której model nie widział przy uczeniu. Na osi poziomej wykresu jest $n/k$: im mniejsze $k$,
tym bardziej elastyczny predyktor, więc $n/k$ gra tę rolę, co stopień wielomianu w Zadaniu 1.14.

In [ ]:
def knn_errors(X_tr, Y_tr, X_te, Y_te, k_values):
    train_err, risk = [], []
    for k in k_values:
        reg = KNeighborsRegressor(n_neighbors=k).fit(X_tr, Y_tr)
        train_err.append(...)   # średnia strata kwadratowa na zbiorze uczącym
        risk.append(...)        # średnia strata kwadratowa na nowej próbie
    return np.array(train_err), np.array(risk)

n = len(Y_tr)
k_values = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30])
train_err, risk = knn_errors(X_tr, Y_tr, X_te, Y_te, k_values)

plt.plot(n / k_values, train_err, "o-", label="błąd na zbiorze uczącym")
plt.plot(n / k_values, risk, "o-", label="ryzyko (nowa próba)")
plt.axhline(sigma ** 2, color="gray", linestyle="--", label="błąd nieredukowalny")
plt.xscale("log")
plt.xlabel("$n/k$")
plt.ylabel("średnia strata kwadratowa")
plt.legend()
plt.show()

Pytania:

1. Dlaczego dla $k=1$ błąd na zbiorze uczącym jest równy zeru (Zadanie 1.6)? Czy mówi to coś o ryzyku?
2. Które $k$ ma najmniejszy błąd na zbiorze uczącym? A które najmniejsze ryzyko oszacowane na nowej próbie?
3. Co przewiduje predyktor dla $k=n$ (Zadanie 3.3)? Ile wynosi wtedy ryzyko w przybliżeniu? (Wskazówka: $\operatorname{Var}\big(\sin(2\pi X)\big)=1/2$.)
4. Porównaj kształt obu krzywych z wykresem dla wielomianów z laboratorium 1.

Wynik zależy od wylosowanej próby uczącej. Powtórz doświadczenie dla 20 prób uczących i narysuj średnie obu krzywych.
Nową próbę możesz zmniejszyć do $10\,000$ obserwacji, żeby obliczenia trwały krócej. Następnie zmień $n$ na 300
(i wartości $k$ tak, by $k=n$ było wśród nich): czy najlepsze $k$ się zmienia? A najlepsze $n/k$?

In [ ]:
...

## Część 3. Klasy niezrównoważone (Zadanie 2.4)

Dwie klasy na prostej: klasa 1 ma rozkład $N(\mu,1)$, klasa 0 -- $N(-\mu,1)$, $\mu=1$. W przykładzie z książki
(rysunek przed zadaniem) klasy były równoliczne. Teraz klasa 1 jest rzadka: $P(Y=1)=\pi_1=0{,}1$, $P(Y=0)=\pi_0=0{,}9$.

### 3.1. Reguła Bayesa i jej błąd

Ze wzoru Bayesa $\eta(x)>1/2$ wtedy i tylko wtedy, gdy $\pi_1\,\varphi(x-\mu)>\pi_0\,\varphi(x+\mu)$, gdzie $\varphi$ to gęstość
$N(0,1)$. Po zlogarytmowaniu dostajemy próg $x^*$: reguła Bayesa wskazuje klasę 1 dla $x>x^*$. Wyznacz $x^*$ i błąd bayesowski, czyli sumę
dwóch składników: $\pi_0\,P\big(N(-\mu,1)>x^*\big)$ (klasa 0 zaklasyfikowana jako 1) i $\pi_1\,P\big(N(\mu,1)<x^*\big)$
(klasa 1 zaklasyfikowana jako 0).
Dystrybuanta $N(0,1)$ to `norm.cdf`, a $P(N(m,1)<t)$ to `norm.cdf(t - m)`.
Porównaj wynik z błędem klasyfikatora trywialnego, który zawsze wskazuje klasę 0,
i z błędem bayesowskim $\Phi(-\mu)$ dla klas równolicznych.

In [ ]:
mu, pi1 = 1.0, 0.1
pi0 = 1 - pi1

threshold = ...
bayes_err = ...
trivial_err = ...
print(f"próg x* = {threshold:.4f},  błąd bayesowski = {bayes_err:.4f},  "
      f"klasyfikator trywialny = {trivial_err:.4f},  klasy równoliczne = {norm.cdf(-mu):.4f}")

### 3.2. Ryzyko liczone całkowaniem

Rozkład znamy, więc ryzyko klasyfikatora możemy policzyć dokładnie, bez zbioru testowego (tak jak w kodzie rysunku
w książce): klasyfikator popełnia błąd na klasie 0 tam, gdzie wskazuje 1, i na klasie 1 tam, gdzie wskazuje 0.
Funkcja `exact_risk` dostaje etykiety, które klasyfikator wskazuje w punktach gęstej siatki, i całkuje
gęstości ważone prawdopodobieństwami a priori po obszarach błędu. Sprawdź ją na regule Bayesa
i na klasyfikatorze trywialnym: wyniki powinny się zgadzać z 3.1.

In [ ]:
grid = np.linspace(-7, 7, 4001)

def exact_risk(labels):
    err0 = np.where(labels == 1, pi0 * norm.pdf(grid + mu), 0.0)
    err1 = ...
    return np.trapezoid(err0 + err1, grid)

print(exact_risk((grid > threshold).astype(int)), exact_risk(np.zeros_like(grid)))

### 3.3. $k$-NN wobec klasyfikatora trywialnego

Losujemy próby uczące rosnącej liczności, uczymy $k$-NN z $k=\sqrt n$ (zaokrąglonym) i liczymy jego ryzyko dokładnie.
Losowanie w dwóch krokach, jak urny w laboratorium 1: najpierw klasa, potem cecha z rozkładu tej klasy.

In [ ]:
def draw_sample(n):
    Y = (rng.random(n) < pi1).astype(int)
    X = ...   # N(mu, 1) dla klasy 1, N(-mu, 1) dla klasy 0
    return X.reshape(-1, 1), Y

sizes = np.array([20, 50, 100, 300, 1000, 3000, 10_000])
n_reps = 25
errors = np.empty((len(sizes), n_reps))
for i, n in enumerate(sizes):
    k = max(1, round(np.sqrt(n)))
    for j in range(n_reps):
        X, Y = draw_sample(n)
        klf = KNeighborsClassifier(n_neighbors=k).fit(X, Y)
        errors[i, j] = ...

means = errors.mean(axis=1)
err_quantiles = np.quantile(errors, [0.1, 0.9], axis=1)
plt.errorbar(sizes, means, yerr=np.abs(err_quantiles - means), marker="o", capsize=3, label="$k$-NN, $k=\\sqrt{n}$")
plt.axhline(bayes_err, color="red", linestyle="--", label="błąd bayesowski")
plt.axhline(trivial_err, color="gray", linestyle=":", label="klasyfikator trywialny")
plt.xscale("log")
plt.xlabel("liczność próby uczącej $n$")
plt.ylabel("ryzyko")
plt.legend()
plt.show()

Pytania:

1. Od jakiego $n$ klasyfikator $k$-NN jest wyraźnie lepszy od trywialnego? Przy małym $n$ górny wąs sięga dokładnie $0{,}1$.
   Co to znaczy? Ile obserwacji klasy 1 jest średnio w próbie $n=20$, ilu sąsiadów bierze wtedy $k$-NN i ilu z nich
   musi należeć do klasy 1, żeby wskazał klasę 1?
2. Różnica między klasyfikatorem trywialnym a regułą Bayesa jest tu mała. Co ją wyznacza? Jak zmienia się, gdy $\mu$ rośnie
   albo gdy $\pi_1$ maleje? Sprawdź, zmieniając parametry w 3.1.
3. Powtórz 3.3 z $k=1$. Czy 1-NN pokonuje klasyfikator trywialny przy którymkolwiek $n$? Porównaj wynik z twierdzeniem Covera--Harta
   z wykładu: granica ryzyka 1-NN to $\mathbb E\big[2\eta(X)(1-\eta(X))\big]$, a oszacowanie z góry -- $2R^*(1-R^*)$.

In [ ]:
...